# 🎙️ Studio Editoriale AI

Questo notebook esegue l'applicazione **Studio Editoriale AI** su Google Colab con accelerazione **GPU T4**, scaricando automaticamente il codice e le voci dal repository GitHub.

### 1. Clonazione del Repository da GitHub
Esegui questa cella per scaricare automaticamente tutti i file del progetto in un click.

In [ ]:
import os

GITHUB_REPO_URL = "https://github.com/emanuelec807/AIStudioEbook.git"
REPO_NAME = "AIStudioEbook"

if not os.path.exists(f"/content/{REPO_NAME}"):
    print(f"⏳ Clonazione del repository {REPO_NAME}...")
    !git clone {GITHUB_REPO_URL} /content/{REPO_NAME}
else:
    print(f"🔄 Aggiornamento repository {REPO_NAME}...")
    %cd /content/{REPO_NAME}
    !git pull

%cd /content/{REPO_NAME}
print(f"\n✅ Repository pronto nella cartella: {os.getcwd()}")

### 2. Installazione Rapida delle Dipendenze (circa 30 secondi)
Esegui questa cella per installare tutte le librerie necessarie tramite pacchetti binari precompilati.

In [ ]:
# 1. Installa Coqui TTS, Kokoro e dipendenze usando ruote binarie precompilate (ultra veloce)
print("⏳ Installazione dipendenze in corso (circa 30 secondi)...")
!pip install -q --prefer-binary coqui-tts kokoro soundfile transformers flask flask-cors pydub EbookLib beautifulsoup4 accelerate requests
!apt-get install -y -qq ffmpeg espeak-ng

print("\n" + "="*50)
print("🎉 TUTTE LE DIPENDENZE INSTALLATE CON SUCCESSO!")
print("👉 Puoi procedere alla Cella 3 (Ollama) oppure direttamente alla Cella 4 per avviare l'app!")
print("="*50)

### 3. (Opzionale) Installazione e Avvio di Ollama per TranslateGemma (12B & 4B)
Se desideri utilizzare la traduzione AI con **TranslateGemma 12B e 4B** su Colab, esegui questa cella.

In [ ]:
import subprocess
import time
import requests
import os

# 1. Installa zstd ed Ollama su Linux
print("⏳ Installazione zstd ed Ollama...")
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Avvia il server Ollama in background
print("⏳ Avvio server Ollama...")
subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=open("ollama.log", "w"))

# Attendi che il server Ollama risponda
print("⏳ Inizializzazione Ollama in corso...")
for _ in range(15):
    try:
        r = requests.get("http://localhost:11434")
        if r.status_code == 200:
            break
    except Exception:
        pass
    time.sleep(1)

# 3. Scarica TranslateGemma 12B e 4B
print("⏳ Download TranslateGemma 12B...")
!ollama pull translategemma:12b
print("⏳ Download TranslateGemma 4B...")
!ollama pull translategemma:4b

print("\n" + "="*50)
print("🎉 OLLAMA & TRANSLATEGEMMA (12B + 4B) PRONTI E ATTIVI!")
print("="*50)

### 4. Avvio dell'Applicazione (Server + Link Pubblico Diretto)
Esegui questa cella per avviare il server ed ottenere subito il **link finale e la password di sicurezza**.

In [ ]:
import subprocess
import time
import requests
import os

# 1. Recupera automaticamente l'IP pubblico per il tunnel
try:
    public_ip = requests.get('https://ipv4.icanhazip.com').text.strip()
except Exception:
    public_ip = "Recupero automatico non riuscito"

# 2. Crea cartelle di output se non esistono
os.makedirs("audiolibri_output", exist_ok=True)
os.makedirs("audiolibriEpub", exist_ok=True)

# Scarica file voce_rif_female.wav se non presente per garantire un preset iniziale
if not os.path.exists("voce_rif_female.wav"):
    print("⏳ Download voce di riferimento preset iniziale...")
    !wget -q https://github.com/DeepMount00/Sibilia-TTS/raw/main/voce_rif_female.wav -O voce_rif_female.wav
    print("✅ Voce preset scaricata!")

# 3. Avvia il server Flask in background con licenza Coqui auto-accettata
print("⏳ Avvio del server Python...")
env = os.environ.copy()
env["COQUI_TOS_AGREED"] = "1"
subprocess.Popen(["python", "server.py"], env=env, stdout=open("flask.log", "w"), stderr=open("flask.log", "w"))

time.sleep(5)
print("✅ Server Flask pronto!")

# 4. Stampa la password di sicurezza e il link diretto
print("\n" + "="*60)
print(f"🔑 PASSWORD DI ACCESSO (COPIA QUESTO IP):  {public_ip}")
print("👉 Clicca sul link '.localtunnel.me' qui sotto, incolla l'IP e premi Submit:")
print("="*60 + "\n")

# 5. Avvia localtunnel direttamente senza prompt (flag --yes)
!npx --yes localtunnel --port 5000